# Transfer Learning med ResNet50

I denna notebook används transfer learning för att klassificera datasettet Signs med den förtränade ResNet50 modellen.

## Vad är transfer learning?

Transfer learning är något man gör när man använder ett redan tränat nätverk så som i det här fallet ResNet50. Och tränar det till ett nytt specifikt problem istället för att träna det från grunden.

Så att man kan dra nytta av någon annas data och optimneringar och nätverksarkitektur och inte bara beräkningskrft.

In [ ]:
import kagglehub

path = kagglehub.dataset_download('maneesh99/signs-detection-dataset')
print(f'Path to dataset files: {path}')

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

train_path = path + '/Signs_Data_Training.h5'

with h5py.File(train_path, 'r') as f:
    print('Nycklar i träningsfilen:', list(f.keys()))
    
    for k in f.keys():
        ds = f[k]
        print(f'  {k}: shape={ds.shape}, dtype={ds.dtype}')

In [ ]:
with h5py.File(train_path, 'r') as f:
    sample_images = f['train_set_x'][:8]
    sample_labels = f['train_set_y'][:8]
    classes = [c.decode() if isinstance(c, bytes) else str(c)
               for c in f['list_classes'][:]]

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for ax, img, lbl in zip(axes, sample_images, sample_labels):
    ax.imshow(img)
    ax.set_title(f'Klass {lbl} ({classes[lbl]})', fontsize=9)
    ax.axis('off')
    
plt.suptitle('Signs Dataset – exempelbilder (64×64 px)', fontsize=12)
plt.tight_layout()
plt.show()

## Förbehandling för ResNet50

ResNet50 är tränad på **ImageNet** med bilder av storleken **224×224 pixlar** och normaliserade med ImageNet-statistik:

- **Mean**: `(0.485, 0.456, 0.406)`  
- **Std**: `(0.229, 0.224, 0.225)`  

Eftersom Signs-bilderna bara är 64×64 px skalar vi **upp** dem till 256 och croppar sedan till 224 – detta är den officiella ResNet50-förbehandlingspipelinen.

Jag lägger även till augmentering (slumpmässig crop, flip, färgvariation) för att öka variationen på bilderna.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from torchvision import models
from torchvision.models import ResNet50_Weights

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 32
print(f'Använder enhet: {DEVICE}')

In [ ]:
class SignsDataset(Dataset):
    '''Laddar Signs-datasetet från en HDF5-fil.'''

    def __init__(self, h5_path, transform=None):
        with h5py.File(h5_path, 'r') as f:
            keys      = list(f.keys())
            image_key = next(k for k in keys if len(f[k].shape) == 4)
            n         = f[image_key].shape[0]
            label_key = next(k for k in keys if k != image_key and f[k].size == n)
            self.images = f[image_key][:].astype('float32') / 255.0
            self.labels = f[label_key][:].flatten().astype('int64')
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        img = torch.from_numpy(self.images[i]).permute(2, 0, 1)  # HWC → CHW
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[i])


class TransformDataset(Dataset):
    '''Wrapper som applicerar en transform på ett subset.'''

    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, i):
        img, label = self.subset[i]
        return self.transform(img), label

In [ ]:
# ImageNet-normalisering (krävs av ResNet50)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize(256, antialias=True),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(256, antialias=True),
    transforms.CenterCrop(224),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('Train transform:\n', train_transform)
print('\nVal/test transform:\n', val_transform)

In [ ]:
full_train = SignsDataset(train_path, transform=None)
val_size   = int(0.2 * len(full_train))
train_size = len(full_train) - val_size
train_set, val_set = random_split(
    full_train, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_set = TransformDataset(train_set, train_transform)
val_set   = TransformDataset(val_set,   val_transform)
test_set  = SignsDataset(path + '/Signs_Data_Testing.h5', transform=val_transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Träning: {len(train_set)} | Validering: {len(val_set)} | Test: {len(test_set)}')

## Modell – ResNet50 med fryst backbone

Vi laddar ResNet50 med de officiella ImageNet-vikterna (`IMAGENET1K_V2`, 80,9% top-1 accuracy).  
Sedan:

1. Fryser alla parametrar för feature extratorn (med `requires_grad = False`)  
2. Ersätter det ursprungliga `fc`-lagret (1 000 klasser) med ett nytt 

Det nya `fc`-lagret initieras slumpmässigt och är det **enda som tränas i fas 1**.  
Resten av nätverket fungerar som en fast **feature extractor**.

### ResNet50-arkitektur (förenklat)

```
conv1 → bn1 → relu → maxpool
  layer1  (3 bottleneck-block)
  layer2  (4 bottleneck-block)
  layer3  (6 bottleneck-block)
  layer4  (3 bottleneck-block)   ← låses upp i fas 2
  avgpool → fc  (ersätter med de 6 handsignal klasserna)
```

Totalt har ResNet50 ~25 miljoner parametrar. Med fryst feature extractor tränar vi om de ~3 000 som finns i `fc`.

In [ ]:
# Ladda ResNet50 med förtränade vikter
weights = ResNet50_Weights.DEFAULT
model   = models.resnet50(weights=weights)

# Frys feature extractor
for param in model.parameters():
    param.requires_grad = False

# Ersätt klassificeringslagret med ett för 6 klasser
model.fc = nn.Linear(model.fc.in_features, 6)
model    = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Totala parametrar:       {total_params:,}')
print(f'Träningsbara parametrar: {trainable_params:,}  '
      f'({100 * trainable_params / total_params:.2f}% av de totala paramtrarna)')

## Fas 1 – Träna klassificeringslagret

Träningen blir betydligt snabbare när vi har fryst feature extracktorn.

Vi använder **Adam** med cosine annealing av inlärningshastigheten.

In [ ]:
EPOCHS = 10
LR     = 1e-3
WD     = 1e-4

# Optimera bara fc-parametrarna
optimizer = optim.Adam(model.fc.parameters(), lr=LR, weight_decay=WD)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}


def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    
    with ctx:
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            if train:
                optimizer.zero_grad()

            out  = model(X)
            loss = criterion(out, y)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(y)
            correct    += (out.argmax(1) == y).sum().item()
            total      += len(y)

    return total_loss / total, correct / total


for epoch in range(1, EPOCHS + 1):
    t_loss, t_acc = run_epoch(train_loader, train=True)
    v_loss, v_acc = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    print(f'Epok {epoch:2d}/{EPOCHS} | '
          f'train_loss={t_loss:.4f} acc={t_acc:.3f} | '
          f'val_loss={v_loss:.4f} acc={v_acc:.3f}')

In [ ]:
_, test_acc = run_epoch(test_loader, train=False)
print(f'Testaccuracy (fas 1 – fryst feature extractor): {test_acc:.3f}')
print(f'Parametrar: {trainable_params:,}')

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('ResNet50 – Signs-Detection (fas 1: fryst feature extractor)',
             fontsize=13, fontweight='bold')

ax1.plot(epochs_range, history['train_loss'], label='Träning')
ax1.plot(epochs_range, history['val_loss'],   label='Validering')
ax1.set_xlabel('Epok')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(epochs_range, history['train_acc'], label='Träning')
ax2.plot(epochs_range, history['val_acc'],   label='Validering')
ax2.set_xlabel('Epok')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

## Fas 2 – Fine-tuning av sista ResNet-blocket

Nu låser vi upp `layer4` (de sista 3 bottleneck-blocken i ResNet50) och tränar det  
tillsammans med `fc`-lagret med en **lägre inlärningshastighet** (`1e-4`).

### Varför låg inlärningshastighet?

De förtränade vikterna i `layer4` är redan väloptimerade. En för hög LR kan **katastrofalt glömma**  
(eng. *catastrophic forgetting*) de inlärda representationerna. Med låg LR finjusterar vi dem  
istället varsamt mot de nya klasserna.

Antal träningsbara parametrar ökar från ~12 000 till ~15 miljoner i denna fas.

In [ ]:
# Lås upp layer4
for param in model.layer4.parameters():
    param.requires_grad = True

trainable_ft = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Träningsbara parametrar (fas 2): {trainable_ft:,}')

FINETUNE_EPOCHS = 5
FINETUNE_LR     = 1e-4

# Ny optimerare med låg LR för alla träningsbara parametrar
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FINETUNE_LR, weight_decay=WD
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS)

history_ft = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, FINETUNE_EPOCHS + 1):
    t_loss, t_acc = run_epoch(train_loader, train=True)
    v_loss, v_acc = run_epoch(val_loader,   train=False)
    scheduler.step()

    history_ft['train_loss'].append(t_loss)
    history_ft['val_loss'].append(v_loss)
    history_ft['train_acc'].append(t_acc)
    history_ft['val_acc'].append(v_acc)

    print(f'Epok {epoch:2d}/{FINETUNE_EPOCHS} | '
          f'train_loss={t_loss:.4f} acc={t_acc:.3f} | '
          f'val_loss={v_loss:.4f} acc={v_acc:.3f}')

_, test_acc_ft = run_epoch(test_loader, train=False)
print(f'Testaccuracy (fas 2 – fine-tuning layer4): {test_acc_ft:.3f}')

In [ ]:
epochs_range_ft = range(1, FINETUNE_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('ResNet50 – Signs-Detection (fas 2: fine-tuning layer4)',
             fontsize=13, fontweight='bold')

ax1.plot(epochs_range_ft, history_ft['train_loss'], label='Träning')
ax1.plot(epochs_range_ft, history_ft['val_loss'],   label='Validering')
ax1.set_xlabel('Epok')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(epochs_range_ft, history_ft['train_acc'], label='Träning')
ax2.plot(epochs_range_ft, history_ft['val_acc'],   label='Validering')
ax2.set_xlabel('Epok')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

## Reflektion – Varför fungerar transfer learning?

> **🤔 Hur kommer det sig att en modell som tränats att känna igen blommor, fordon, etc.**  
> **snabbare blir bra på att känna igen just dina kategorier av objekt, även om nätverket**  
> **inte har tränats på just sådan data tidigare?**

Svaret ligger i **hur djupa neurala nätverk representerar information**:

### Hierarkiska representationer

Ett djupt CNN lär sig representationer på flera abstraktionsnivåer:

| Lager | Vad det lär sig |
|-------|-----------------|
| Tidiga (`conv1`, `layer1`) | Primitiva strukturer: kanter, hörn, färggradienter |
| Mellersta (`layer2`, `layer3`) | Texturer, mönster, enkla former (cirklar, kurvor) |
| Sena (`layer4`) | Komplexa mönster, objektdelar och sammansättningar |
| `fc`-lagret | Klasspecifika beslut |

### Generaliserbarhet

De **tidiga och mellersta lagrens representationer är domänoberoende**: kanter, texturer och  
former är lika relevanta oavsett om man klassificerar hundar, handtecken eller trafikskyltar.

ResNet50 har tränats på 1,2 miljoner bilder med enorm variation, vilket gjort att dess  
feature extractor är exceptionellt bra på att **extrahera meningsfull information ur bilder**.  
Denna förmåga är direkt överförbar till nya domäner.

### Varför snabbare träning?

Utan transfer learning måste nätverket **lära sig allt från grunden** – att känna igen kanter,  
former och texturer OCH de specifika klasserna – utifrån bara 864 träningsbilder. Med transfer  
learning börjar vi redan med en kraftfull feature extractor och behöver **bara anpassa  
de klasspecifika delarna**.

### Sammanfattning

Det handlar inte om att nätverket har "sett handtecken" förut. Det handlar om att det har lärt  
sig **universella visuella byggstenar** som är relevanta för i stort sett alla bildklassificeringsproblem.  
Dessa byggstenar – kanter, former, texturer – är värdefulla oavsett domän och gör att vi kan  
uppnå hög noggrannhet även med lite träningsdata.

## Slututvärdering efter fine-tuning

Kör detta efter fas 2. Den ger per-klass-resultat och confusion matrix för den finjusterade ResNet50-modellen.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt

# Testutvärdering + prediktioner för finjusterad modell
model.eval()
all_preds, all_labels = [], []
test_loss, test_correct, test_total = 0.0, 0, 0

with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = model(X)
        loss = criterion(logits, y)

        preds = logits.argmax(dim=1)
        test_loss += loss.item() * X.size(0)
        test_correct += (preds == y).sum().item()
        test_total += y.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

test_loss /= test_total
test_acc_finetuned = test_correct / test_total
print(f"Testresultat ResNet50 fine-tuned: loss={test_loss:.3f}, accuracy={test_acc_finetuned:.3f}")

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
classes = sorted(np.unique(all_labels))

print("\nPer-klass accuracy:")
for cls in classes:
    mask = all_labels == cls
    cls_acc = (all_preds[mask] == all_labels[mask]).mean()
    print(f"  Klass {cls}: {cls_acc:.3f}  (n={mask.sum()})")

cm = confusion_matrix(all_labels, all_preds, labels=classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, values_format="d")
ax.set_title("ResNet50 fine-tuned – confusion matrix på testdata")
plt.show()


## Slutsats och gämförelse mellan min egen CNN och ResNet50

I part 3 jämfördes tre modeller på Signs Detection Dataset: en egen CNN tränad från början, ResNet50 med fryst feature extractor och ResNet50 efter fine-tuning av sista blocket.

Min egen CNN, MediumCNN, tränades helt från slumpmässigta vikter. Den har ungefär 8,4 miljoner träningsbara parametrar, eftersom alla lager tränas från början. Modellen är relativt liten jämfört med ResNet50, men den måste själv lära sig både enkla bildfeatures, som kanter och färger, och mer avancerade former som skiljer handtecknen åt. Därför kräver den mer data och fler epoker för att nå hög accuracy.

ResNet50 med fryst feature extractor gav 87,5 % testaccuracy. Här tränades endast det sista klassificeringslagret, alltså 12 294 träningsbara parametrar, vilket är väldigt lite jämfört med hela modellen. Fördelen är att träningen blir snabb och risken för överanpassning minskar, eftersom nästan hela modellen redan är färdigtränad på ImageNet. Nackdelen är att modellen inte kan anpassa sina djupare features särskilt mycket till just handteckens-datasetet.

Efter fine-tuning av layer4 ökade antalet träningsbara parametrar till cirka 15 miljoner, och testaccuracy steg till 97,5 %. Detta blev det bästa resultatet. Fine-tuning fungerar bättre eftersom modellen fortfarande använder generella features från ImageNet, men de sista lagren får anpassas till de specifika klasserna i Signs-datasetet. Valideringsaccuracy låg nära träningsaccuracy och testresultatet var högt, så modellen verkar inte over fitted (eller hur man nu säger på svengelska), även om datasetet är ganska litet.

Transfer learning fungerar bra här eftersom handteckensbilder fortfarande består av visuella strukturer som ResNet50 redan är bra på att känna igen: kanter, former, konturer, färgskillnader och objektliknande mönster. Datasetet är dessutom litet, med bara 864 träningsbilder efter split, vilket gör det svårt för en egen CNN att lära sig robusta features från grunden. Därför får ResNet50 en tydlig fördel. Den frysta modellen är snabbast och mest resurssnål, men fine-tuning ger bäst accuracy eftersom den kombinerar förtränad generell bildförståelse med anpassning till det nya problemet.

| Modell              | Testaccuracy | Träningsbara parametrar | Kommentar                                  |
| ------------------- | ------------ | ----------------------- | ------------------------------------------ |
| Egen MediumCNN      | 86,7 %       | ca 8,4 miljoner         | Tränas från grunden                        |
| ResNet50 fryst      | 87,5 %       | 12 294                  | Snabb, låg risk för overfitting            |
| ResNet50 fine-tuned | 97,5 %       | ca 15,0 miljoner        | Bäst resultat, mer anpassad till datasetet |
